# Analysis Queries

**Purpose:** Demonstrate analytical insights from gold tables using SQL. These queries can be used for dashboards, reports, or ad-hoc analysis.

**Data Sources:**
* `workspace.default.movies_silver` - For detailed movie-level analysis
* `workspace.default.movies_gold_by_genre` - For genre-level metrics
* `workspace.default.movies_gold_by_decade` - For time-series trends
* `workspace.default.movies_bronze_genres` - For genre lookups

**Analysis Categories:**
1. **Genre Analysis**: Which genres produce the highest-rated content?
2. **Trend Analysis**: How have movie metrics evolved over decades?
3. **Top Lists**: What are the highest-rated movies?
4. **Correlation Analysis**: Does popularity correlate with quality?
5. **Diversity Analysis**: International cinema representation

**Execution Order:** Run this notebook after all gold tables are created (03-gold-aggregations).

In [0]:
%sql
-- Query 1: Top Genres by Rating
-- Shows which genres consistently produce the highest-rated movies
-- Requires minimum 50 movies per genre for statistical significance

WITH movies_exploded AS (
  SELECT
    movie_id,
    vote_average,
    vote_count,
    CAST(genre_id_exploded AS INT) AS genre_id
  FROM
    workspace.default.movies_silver
  CROSS JOIN
    LATERAL explode(from_json(genre_ids, 'array<int>')) AS genre_table(genre_id_exploded)
  WHERE
    vote_count >= 50
)
SELECT
  g.genre_name,
  COUNT(DISTINCT m.movie_id) AS movie_count,
  ROUND(AVG(m.vote_average), 2) AS avg_rating,
  ROUND(MIN(m.vote_average), 2) AS min_rating,
  ROUND(MAX(m.vote_average), 2) AS max_rating,
  ROUND(STDDEV(m.vote_average), 2) AS rating_stddev
FROM
  movies_exploded m
INNER JOIN
  workspace.default.movies_bronze_genres g
  ON m.genre_id = g.genre_id
GROUP BY
  g.genre_name
HAVING
  COUNT(DISTINCT m.movie_id) >= 50
ORDER BY
  avg_rating DESC,
  movie_count DESC;

In [0]:
%sql
-- Query 2: Decade Trends
-- Shows how movie production and quality metrics have evolved over time
-- Compares decade-over-decade changes in volume, ratings, and engagement

WITH decade_metrics AS (
  SELECT
    release_decade,
    total_movies,
    avg_rating,
    avg_popularity,
    total_votes,
    avg_votes_per_movie,
    LAG(total_movies) OVER (ORDER BY release_decade) AS prev_decade_movies,
    LAG(avg_rating) OVER (ORDER BY release_decade) AS prev_decade_rating
  FROM
    workspace.default.movies_gold_by_decade
)
SELECT
  release_decade,
  total_movies,
  avg_rating,
  avg_popularity,
  avg_votes_per_movie,
  total_movies - prev_decade_movies AS movie_count_change,
  ROUND(avg_rating - prev_decade_rating, 2) AS rating_change,
  ROUND(
    ((total_movies - prev_decade_movies) * 100.0) / NULLIF(prev_decade_movies, 0),
    1
  ) AS pct_change_in_volume
FROM
  decade_metrics
ORDER BY
  release_decade;

In [0]:
%sql
-- Query 3: Top Rated Movies
-- Shows the highest-rated movies in the database
-- Requires minimum 500 votes to ensure rating credibility and avoid bias from small samples

SELECT
  title,
  release_year,
  vote_average AS rating,
  vote_count AS votes,
  popularity,
  original_language
FROM
  workspace.default.movies_silver
WHERE
  vote_count >= 500
ORDER BY
  vote_average DESC,
  vote_count DESC
LIMIT 25;

In [0]:
%sql
-- Query 4: Rating vs Popularity - Do popular movies get better ratings?
-- Bucket movies by popularity and compare average ratings

WITH popularity_buckets AS (
  SELECT
    title,
    vote_average,
    popularity,
    CASE
      WHEN popularity < 50 THEN 'Low (<50)'
      WHEN popularity < 150 THEN 'Medium (50-150)'
      WHEN popularity < 300 THEN 'High (150-300)'
      ELSE 'Very High (300+)'
    END AS popularity_tier
  FROM
    workspace.default.movies_silver
  WHERE
    vote_count >= 50
)
SELECT
  popularity_tier,
  COUNT(*) AS movie_count,
  ROUND(AVG(vote_average), 2) AS avg_rating,
  ROUND(MIN(vote_average), 2) AS min_rating,
  ROUND(MAX(vote_average), 2) AS max_rating,
  ROUND(STDDEV(vote_average), 2) AS rating_stddev
FROM
  popularity_buckets
GROUP BY
  popularity_tier
ORDER BY
  CASE popularity_tier
    WHEN 'Low (<50)' THEN 1
    WHEN 'Medium (50-150)' THEN 2
    WHEN 'High (150-300)' THEN 3
    WHEN 'Very High (300+)' THEN 4
  END;

In [0]:
%sql
-- Query 5: Language Diversity - International cinema landscape
-- Shows the most represented languages in modern cinema (post-2010)

SELECT
  original_language,
  COUNT(*) AS movie_count,
  ROUND(AVG(vote_average), 2) AS avg_rating,
  ROUND(AVG(popularity), 2) AS avg_popularity,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage_of_total
FROM
  workspace.default.movies_silver
WHERE
  release_year >= 2010
  AND vote_count >= 20
GROUP BY
  original_language
HAVING
  COUNT(*) >= 10
ORDER BY
  movie_count DESC
LIMIT 15;